In [0]:
"""
Helper notebook for transitioning the MLflow model alias from 'challenger' to 'champion' in the Model Registry after validation.
This notebook is intended to be run as part of a multi-task job following the Train.py notebook.

Workflow:
- Receives parameters for environment ('env') and model name ('model_name') via Databricks widgets.
- Constructs the model URI for the 'challenger' alias.
- Deploys the model by transitioning the alias from 'challenger' to 'champion' for the specified environment using the deploy() function.
- Prints a success message upon completion.

Parameters:
    env (str): Name of the current environment for model deployment. Determines the target stage ('dev', 'staging', 'prod').
    model_name (str): Name of the MLflow model to deploy. Used to construct the model URI.

Notes:
- The model URI must be in the format "models:/<name>@challenger".
- The notebook assumes a preceding task has set the required parameters.
- After validation, the model alias is transitioned from 'challenger' to 'champion'.
- In production, it is recommended to include a model validation step before alias transition.

References:
- MLflow Model Registry: https://www.mlflow.org/docs/latest/model-registry.html#fetching-an-mlflow-model-from-the-model-registry
- Databricks task values: https://docs.databricks.com/dev-tools/databricks-utils.html
"""

In [0]:
# List of input args needed to run the notebook as a job.
# Provide them via DB widgets or notebook arguments.
#
# Name of the current environment
dbutils.widgets.dropdown("env", "dev", ["dev", "staging", "prod"], "Environment Name")
dbutils.widgets.text("model_name","telco_churn_model", "Model Name")
dbutils.widgets.text(
    "username",
    "",
    label="Username",
)
dbutils.widgets.text("catalog_name","mlops_dbx_talk_dev", "Catalog Name")

In [0]:
import sys
sys.path.append('../..')
from model_deployment.deploy import deploy

username = dbutils.widgets.get("username")
if username=="" or username is None:
    raise Exception("Provide the username")
catalog_name = dbutils.widgets.get("catalog_name")

model_name = f"{catalog_name}.{username}.{dbutils.widgets.get('model_name')}"

model_uri = f"models:/{model_name}@challenger"
env = dbutils.widgets.get("env")
assert env != "None", "env notebook parameter must be specified"
assert model_uri != "", "model_uri notebook parameter must be specified"
deploy(model_uri, env)

In [0]:
print(
    f"Successfully completed model deployment for {model_uri}"
)